# Hospital Readmission Prediction

## Case Study 1
**Goal:** Predict whether a patient will be readmitted within 30 days of discharge.

Uses:
- Logistic Regression
- L2 Regularization
- ROC-AUC evaluation
- Threshold analysis
- False Negative vs False Positive discussion


Importing Libraries

In [3]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    roc_auc_score,
    confusion_matrix,
    classification_report
)

Load datasets

In [4]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    roc_auc_score,
    confusion_matrix,
    classification_report
)


Create Target Values

In [5]:
df["readmit_30"] = (df["readmitted"] == "<30").astype(int)

print("30-Day Readmission Rate:",
      round(df["readmit_30"].mean() * 100, 2), "%")

30-Day Readmission Rate: 11.39 %


Select Features

In [6]:
features = [
    "time_in_hospital",
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient",
    "number_diagnoses"
]

X = df[features]
y = df["readmit_30"]

print("Features Used:")
for feature in features:
    print("-", feature)


Features Used:
- time_in_hospital
- num_lab_procedures
- num_procedures
- num_medications
- number_outpatient
- number_emergency
- number_inpatient
- number_diagnoses


Split Training and Testing Data

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training Data:", X_train.shape)
print("Testing Data:", X_test.shape)


Training Data: (79474, 8)
Testing Data: (19869, 8)


Scale the Features

In [8]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("Data Scaling Complete!")


Data Scaling Complete!


Create Logistic Regression Model

In [24]:
model = LogisticRegression(
    penalty="l2",
    C=1.0,
    class_weight="balanced",
    max_iter=1000
)

print("Model Created!")
print("Regularization: L2")


Model Created!
Regularization: L2


Train the Model

In [21]:
model.fit(X_train, y_train)

print("Model Training Complete!")


Model Training Complete!


Predict Readmission Probabilities

In [22]:
probs = model.predict_proba(X_test)[:, 1]

print("First 10 Readmission Probabilities:")
print(probs[:10])


First 10 Readmission Probabilities:
[0.40714358 0.4641047  0.68619646 0.46873041 0.47449922 0.48931534
 0.44503303 0.52603696 0.37013569 0.46728315]


Evaluate Using ROC-AUC

In [23]:
auc = roc_auc_score(y_test, probs)

print("=" * 50)
print("ROC-AUC Score:", round(auc, 4))
print("=" * 50)


ROC-AUC Score: 0.6347


Threshold Analysis

In [20]:
for threshold in [0.50, 0.35, 0.25]:

    predictions = (probs >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_test,
        predictions
    ).ravel()

    sensitivity = tp / (tp + fn)
    specificity = tn / (tn + fp)

    print("\n" + "=" * 50)
    print("THRESHOLD:", threshold)
    print("=" * 50)

    print("True Negatives :", tn)
    print("False Positives:", fp)
    print("False Negatives:", fn)
    print("True Positives :", tp)

    print("Sensitivity:", round(sensitivity, 3))
    print("Specificity:", round(specificity, 3))



THRESHOLD: 0.5
True Negatives : 12513
False Positives: 5093
False Negatives: 1185
True Positives : 1078
Sensitivity: 0.476
Specificity: 0.711

THRESHOLD: 0.35
True Negatives : 582
False Positives: 17024
False Negatives: 26
True Positives : 2237
Sensitivity: 0.989
Specificity: 0.033

THRESHOLD: 0.25
True Negatives : 0
False Positives: 17606
False Negatives: 0
True Positives : 2263
Sensitivity: 1.0
Specificity: 0.0


Classification Report

In [14]:
predictions = (probs >= 0.50).astype(int)

print(classification_report(y_test, predictions))


              precision    recall  f1-score   support

           0       0.91      0.71      0.80     17606
           1       0.17      0.48      0.26      2263

    accuracy                           0.68     19869
   macro avg       0.54      0.59      0.53     19869
weighted avg       0.83      0.68      0.74     19869



Final Summary

In [15]:
print("=" * 60)
print("FINAL RESULTS")
print("=" * 60)

print("Model: Logistic Regression")
print("Regularization: L2")
print("ROC-AUC:", round(auc, 4))
print("Target: 30-Day Hospital Readmission")

print("\nClinical Cost:")
print("False Negative = High-risk patient is missed")
print("False Positive = Unnecessary intervention")


FINAL RESULTS
Model: Logistic Regression
Regularization: L2
ROC-AUC: 0.6347
Target: 30-Day Hospital Readmission

Clinical Cost:
False Negative = High-risk patient is missed
False Positive = Unnecessary intervention
